In [1]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
import numpy as np
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import yaml
import graphviz
from sklearn import tree

In [2]:
import joblib
import pickle

In [3]:
def Getdata():
    
    action_table=pd.read_csv('./action_table.csv').values[:,1:]
    dqn_data=np.load('./DQN/Results/0.npy',allow_pickle=True).tolist()
    dqn_s=np.array(dqn_data['state'])[1:,:]
    dqn_a=np.array(dqn_data['action'])
    
    ppo_data=np.load('./PPO/Results/0.npy',allow_pickle=True).tolist()
    ppo_s=np.array(ppo_data['state'])[1:,:]
    ppo_a=np.array(ppo_data['action'])[:,0,:]
    
    
    for idx in range(1,1000):
        dqn_data=np.load('./DQN/Results/'+str(idx)+'.npy',allow_pickle=True).tolist()
        dqn_s=np.concatenate((dqn_s,np.array(dqn_data['state'])[1:,:]),axis=0)
        dqn_a=np.concatenate((dqn_a,np.array(dqn_data['action'])),axis=0)
        
        ppo_data=np.load('./PPO/Results/'+str(idx)+'.npy',allow_pickle=True).tolist()
        ppo_s=np.concatenate((ppo_s,np.array(ppo_data['state'])[1:,:]),axis=0)
        ppo_a=np.concatenate((ppo_a,np.array(ppo_data['action'])[:,0,:]),axis=0)
    
    print(dqn_s.shape,dqn_a.shape,ppo_s.shape,ppo_a.shape)
    return dqn_s,dqn_a,ppo_s,ppo_a

In [4]:
def get_y(data):
    #从数据中泵的策略给出分类的y
    y=np.zeros((data.shape[0],1))
    output = {}
    t = 0
    for i in range(3):
        for j in range(3):
            for k in range(3):
                for m in range(2):
                    output[str(i)+','+str(j)+','+str(k)+','+str(m)]=t
                    t+=1
    
    for line in range(y.shape[0]):
        i1,i2,i3,i4=int(np.sum(data[line,0:2])),int(np.sum(data[line,2:4])),int(np.sum(data[line,4:6])),int(np.sum(data[line,6:7]))
        y[line]=output[str(i1)+','+str(i2)+','+str(i3)+','+str(i4)]
    
    class_names=[]
    for it in output:
        class_names.append('Class '+str(output[it]))
    
    return y,class_name

In [5]:
def get_y2(data):
    #从数据中泵的策略给出分类的y
    action_table=pd.read_csv('./action_table.csv').values[:,1:]
    
    y=np.zeros((data.shape[0],1))
    for line in range(y.shape[0]):
        y[line]=np.where((action_table==data[line,:]).all(axis=1))
        
    class_names=[]
    for it in range(128):
        class_names.append('Class '+str(it))
    return y,class_names

In [6]:
dqn_s,dqn_a,ppo_s,ppo_a=Getdata()
config = yaml.load(open("chaohu.yaml"), yaml.FullLoader)
#根据得到的决策树，把几个筛选条件节点名字给出，便于画图
feature_names=[]
for i in range(len(config['states'])):
    if i == 8:
        feature_names.append('WS02006229 flow')
    elif i == 9:
        feature_names.append('WS02006116 flow')
    elif i==10:
        feature_names.append('YS02001907 flow')
    elif i==11:
        feature_names.append('YS02001649 flow')
    elif i==12:
        feature_names.append('CC-1 flow')
    elif i==13:
        feature_names.append('CC-2 flow')
    elif i==14:
        feature_names.append('JK-1 flow')
    elif i==15:
        feature_names.append('JK-2 flow')
    elif i==16:
        feature_names.append('WSC flow')
    elif i==17:
        feature_names.append('Rain intensity')
    else:
        feature_names.append(config['states'][i][0])
        


(95008, 18) (95008, 7) (95010, 18) (95010, 7)


In [8]:
dqn_y,class_names=get_y2(dqn_a)
tree_dqn=DecisionTreeClassifier(criterion="gini",min_impurity_decrease=1e-3,ccp_alpha=0.0001)
tree_dqn.fit(dqn_s,dqn_y)
dot_dqn = tree.export_graphviz(tree_dqn, out_file=None, 
                         feature_names=feature_names, 
                         class_names=class_names,
                         filled=True, rounded=True,  
                         special_characters=True)  
graph = graphviz.Source(dot_dqn)
graph.render('./dqn_tree_final')
joblib.dump(tree_dqn, "dqntree_model_final.m")

C:\Users\chongtm\AppData\Local\Temp\ipykernel_14720\314570240.py:7: DeprecationWarning: setting an array element with a sequence. This was supported in some cases where the elements are arrays with a single element. For example `np.array([1, np.array([2])], dtype=int)`. In the future this will raise the same ValueError as `np.array([1, [2]], dtype=int)`.
  y[line]=np.where((action_table==data[line,:]).all(axis=1))


['dqntree_model_final.m']

In [9]:
ppo_y,class_name=get_y2(ppo_a)
tree_ppo=DecisionTreeClassifier(criterion="gini",min_impurity_decrease=1e-3,ccp_alpha=0.0001)
tree_ppo.fit(ppo_s,ppo_y)
dot_ppo = tree.export_graphviz(tree_ppo, out_file=None, 
                         feature_names=feature_names,  
                         class_names=class_names,
                         filled=True, rounded=True,  
                         special_characters=True)  
graph = graphviz.Source(dot_ppo)
graph.render('./ppo_tree_final')
joblib.dump(tree_ppo, "ppotree_model_final.m")

C:\Users\chongtm\AppData\Local\Temp\ipykernel_14720\314570240.py:7: DeprecationWarning: setting an array element with a sequence. This was supported in some cases where the elements are arrays with a single element. For example `np.array([1, np.array([2])], dtype=int)`. In the future this will raise the same ValueError as `np.array([1, [2]], dtype=int)`.
  y[line]=np.where((action_table==data[line,:]).all(axis=1))


['ppotree_model_final.m']

# Class names 里面的每个对应哪个控制策略

## 找到满足条件的看输出是什么

In [10]:
def find_class(all_ind,data,tree_model):
    #给出分支条件，查看对应的action class的编号
    width=10
    for it in all_ind:
        ind=all_ind[it]['ind']
        ind_index=all_ind[it]['ind_index']
        ind_levels=all_ind[it]['ind_levels']
        tem=data.copy()
        for i in range(len(ind_levels)):
            if ind_index[i]=='le':
                te=tem[tem[ind[i]]<=ind_levels[i]]
            elif ind_index[i]=='me':
                te=tem[tem[ind[i]]>ind_levels[i]]
            else:
                te=tem[tem[ind[i]]==ind_levels[i]]
            if te.shape[0]<width:
                break
            else:
                tem=te
                
        y=tree_model.predict(tem.values[0,:-1].reshape(1, -1))
        print('The Class '+it+' is '+str(y[0]))

## 所有可能的action如下

In [11]:
unique,count=np.unique(dqn_y,return_counts=True)
data_count=dict(zip(unique,count))
print(data_count)

{0.0: 112, 2.0: 6024, 20.0: 88746, 21.0: 126}


In [12]:
unique,count=np.unique(ppo_y,return_counts=True)
data_count=dict(zip(unique,count))
print(data_count)

{0.0: 68979, 2.0: 3898, 4.0: 853, 6.0: 9074, 16.0: 1415, 18.0: 1502, 20.0: 7289, 22.0: 2000}


In [13]:
dqn_data=pd.DataFrame(np.concatenate((dqn_s,dqn_y),axis=1),columns=feature_names+['action'])
ppo_data=pd.DataFrame(np.concatenate((ppo_s,ppo_y),axis=1),columns=feature_names+['action'])

# 各个state编号如下

In [14]:
for i in range(len(feature_names)):
    print(i,feature_names[i])

0 CC-storage
1 JK-storage
2 WS02006229
3 WS02006116
4 WS02006235
5 WS02006251
6 YS02001907
7 YS02001649
8 WS02006229 flow
9 WS02006116 flow
10 YS02001907 flow
11 YS02001649 flow
12 CC-1 flow
13 CC-2 flow
14 JK-1 flow
15 JK-2 flow
16 WSC flow
17 Rain intensity


In [15]:
# PPO条件
all_ind={}
#Class0: WS02006251 ≤ 0.396, WSC flow > 376.883, WS02006116 > 0.454, WSC flow ≤ 554.031
ind=ppo_data.columns[[5,16,3,16]]
ind_index=['le','me','me','le']
ind_levels=[0.396,376.883,0.454,554.031]
all_ind['0']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class1: WS02006251 ≤ 0.396, WSC flow > 376.883, WS02006116 ≤ 0.454
ind=ppo_data.columns[[5,16,3]]
ind_index=['le','me','le']
ind_levels=[0.396,376.883,0.454]
all_ind['1']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class2: WS02006251 > 0.396, YS02001649 > 0.28, CC-2 flow > 341.5, WS02006116 flow ≤ 1226.83, 
#        YS02001649 flow > 1550.418, CC-storage > 1.471, JK-storage ≤ 3.658
ind=ppo_data.columns[[5,7,13,9,11,0,1]]
ind_index=['me','me','me','le','me','me','le']
ind_levels=[0.396,0.28,341.5,1226.83,1550.418,1.471,3.658]
all_ind['2']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class3: WS02006251 > 0.396, YS02001649 ≤ 0.28, WSC flow > 651.916, JK-storage ≤ 1.715, CC-1 flow ≤ 341.5
ind=ppo_data.columns[[5,7,16,1,12]]
ind_index=['me','le','me','le','le']
ind_levels=[0.396,0.28,651.916,1.715,341.5]
all_ind['3']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class4: WS02006251 > 0.396, YS02001649 > 0.28, CC-2 flow > 341.5, WS02006116 flow > 1226.83, CC-storage > 1.625
ind=ppo_data.columns[[5,7,13,9,0]]
ind_index=['me','me','me','me','me']
ind_levels=[0.396, 0.28, 341.5, 1226.83, 1.625]
all_ind['4']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class5: WS02006251 > 0.396, YS02001649 > 0.28, CC-2 flow < 341.5, WSC flow > 1159.091, CC-storage ≤ 1.336, WSC flow > 1419.051
ind=ppo_data.columns[[5,7,13,16,0,16]]
ind_index=['me','me','le','me','le','me']
ind_levels=[0.396, 0.28, 341.5, 1159.091, 1.336, 1419.051]
all_ind['5']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class6: WS02006251 > 0.396, YS02001649 > 0.28, CC-2 flow > 341.5, WS02006116 flow ≤ 1226.83, 
#        YS02001649 flow > 1550.418, CC-storage > 1.471, JK-storage > 3.658
ind=ppo_data.columns[[5,7,13,11,0,1]]
ind_index=['me','me','me','le','me','me','me']
ind_levels=[0.396, 0.28, 341.5, 1226.83,1550.418, 1.471,3.658]
all_ind['6']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class7: WS02006251 > 0.396, YS02001649 ≤ 0.28, WSC flow ≤ 651.916, YS02001907 ≤ 0.273
ind=ppo_data.columns[[5,7,16,6]]
ind_index=['me','le','le','le']
ind_levels=[0.396,0.28,651.916,0.273]
all_ind['7']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}


find_class(all_ind,ppo_data,tree_ppo)



The Class 0 is 0.0
The Class 1 is 2.0
The Class 2 is 4.0
The Class 3 is 6.0
The Class 4 is 16.0
The Class 5 is 18.0
The Class 6 is 20.0
The Class 7 is 22.0


In [16]:
# DQN条件
all_ind={}
#Class2: WS02006116 flow<0.983, RG<20.373
ind=dqn_data.columns[[9,17]]
ind_index=['le','le']
ind_levels=[983.001,20.373]
all_ind['2']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class1: WS02006116 flow<0.983, RG>20.373, WSC>840.983
ind=dqn_data.columns[[9,17,16]]
ind_index=['le','me','me']
ind_levels=[983,20.373,840.983]
all_ind['1']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class0: WS02006116 flow>983, RG>18.87, WSC>790.497, CC-storage<1.392, JK-storage<3.937, RG<26.177, CC-storage>1.244, 
ind=dqn_data.columns[[9,17,16,0,1,17,0]]
ind_index=['me','me','me','le','le','le','me']
ind_levels=[983.001,18.87,790.497,1.392,3.937,26.177,1.244]
all_ind['0']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

#Class3: WS02006116 flow>983, RG>18.87, WSC>790.497, CC-storage>1.392, YS02001907 flow>662.828, 
#        CC-storage<1.48, JK-storage<3.644, JK-storage>3.331, WS02006235>1.904, 
ind=dqn_data.columns[[9,17,16,0,10,0,1,1,4]]
ind_index=['me','me','me','me','me','le','le','me','me']
ind_levels=[983.001,18.87,790.497,1.392,662.828,1.48,3.644,3.331,1.904]
all_ind['3']={'ind':ind,'ind_index':ind_index,'ind_levels':ind_levels}

find_class(all_ind,dqn_data,tree_dqn)

The Class 2 is 20.0
The Class 1 is 2.0
The Class 0 is 2.0
The Class 3 is 20.0


# 模型读取

In [56]:
test_dqntree=joblib.load("dqntree_model_final.m")
test_ppotree=joblib.load("ppotree_model_final.m")